In [1]:
import pandas as pd
import numpy as np
import glob
import os
import shutil
import matplotlib.pyplot as plt

import mne
import pyprep 

from mne.preprocessing import read_ica
from mne_icalabel import label_components
from mne_bids.write import _write_json

from mne_bids import BIDSPath, read_raw_bids, get_entity_vals, write_raw_bids

%matplotlib widget

In [2]:
# Define the path to the BIDS root directory
bids_root = '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/sourcedataRest'

# Define the path to the output directory (derivatives)
out_path = '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline'

# Get all subjects and sessions from the BIDS dataset
subjects = get_entity_vals(bids_root, 'subject')
sessions = get_entity_vals(bids_root, 'session')

# Define the task
task = 'restEyesClosed'

In [3]:
# Loop through each subject and session
for subject in subjects:
    subject_id = f"sub-{subject}"

    for session in sessions:
        session_id = f"ses-{session}"
        session_path = os.path.join(out_path, subject_id, session_id, 'eeg')

        # Define file paths for the current session
        ica_path = os.path.join(session_path, f"{subject_id}_{session_id}_task-{task}_proc-ica_ica.fif")
        tsv_path = os.path.join(session_path, f"{subject_id}_{session_id}_task-{task}_proc-ica_components.tsv")
        json_path = os.path.join(session_path, f"{subject_id}_{session_id}_task-{task}_proc-ica_components.json")
        epo_path = os.path.join(session_path, f"{subject_id}_{session_id}_task-{task}_epo.fif")

        # Check if the ICA file exists
        if not os.path.isfile(ica_path):
            #print(f"ICA file not found: {ica_path}")
            continue

        try:
            # Read ICA object and epochs
            ica = read_ica(ica_path)
            epochs = mne.read_epochs(epo_path, preload=True)

            # Ensure epochs are rereferenced to average
            #if not epochs.info['custom_ref_applied']:
            #    epochs.set_eeg_reference('average', projection=False)

            # Apply ICLabel to label components
            out = label_components(epochs, ica, method='iclabel')
            print(f"Labels for {subject_id}, {session_id}: {out['labels']}")

            # Identify bad components
            bad_components = [i for i, label in enumerate(out['labels']) if label not in ['brain', 'other']]
            ica.exclude.extend(bad_components)

            # Save updated ICA object
            ica.save(ica_path, overwrite=True)

            # Update the TSV file
            tsv_data = pd.read_csv(tsv_path, sep='\t')
            tsv_data['status'] = ['bad' if i in bad_components else 'good' for i in range(len(out['labels']))]
            tsv_data['status_description'] = [f'mne_icalabel::{label}' if i in bad_components else 'n/a' for i, label in enumerate(out['labels'])]
            tsv_data['ic_type'] = out['labels']
            tsv_data['annotate_author'] = 'Alina'
            tsv_data['annotate_method'] = 'mne_icalabel'
            tsv_data.to_csv(tsv_path, sep="\t", index=False, encoding="utf-8")

            # Update the JSON file
            component_json = {
                "annotate_method": "Method used for annotating components (e.g., manual, iclabel)",
                "annotate_author": "The name of the person who ran the annotation",
                "ic_type": "The type of annotation (e.g., brain, muscle artifact, eye blink, etc.)",
            }
            _write_json(json_path, component_json, overwrite=True)

            print(f"Processed {subject_id}, {session_id}")

        except Exception as e:
            print(f"Failed to process {subject_id}, {session_id}: {e}")

print("Finished processing all subjects and sessions.")

Reading /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20180209/eeg/sub-NZ000052_ses-20180209_task-restEyesClosed_proc-ica_ica.fif ...
    Read a total of 1 projection items:
        Average EEG reference (1 x 54)  idle
Now restoring ICA solution ...
Ready.
Reading /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20180209/eeg/sub-NZ000052_ses-20180209_task-restEyesClosed_epo.fif ...
    Read a total of 1 projection items:
        Average EEG reference (1 x 54)  idle
    Found the data of interest:
        t =       0.00 ...    3000.00 ms
        0 CTF compensation matrices available
Not setting metadata
431 matching events found
No baseline correction applied
Created an SSP operator (subspace dimension = 1)
1 projection items activated


/tmp/ipykernel_406678/4062126790.py:30: RuntimeWarning: The provided Epochs instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  out = label_components(epochs, ica, method='iclabel')


Labels for sub-NZ000052, ses-20180209: ['brain', 'brain', 'heart beat', 'brain', 'brain', 'brain', 'eye blink', 'brain', 'brain', 'brain', 'brain', 'brain', 'brain', 'brain', 'brain', 'channel noise', 'eye blink', 'brain', 'heart beat', 'brain', 'channel noise', 'other', 'brain', 'brain', 'channel noise', 'brain', 'brain', 'brain', 'other', 'channel noise', 'other', 'other', 'other', 'channel noise', 'brain', 'eye blink', 'brain', 'brain', 'brain', 'brain', 'brain', 'brain', 'brain', 'other', 'brain', 'channel noise', 'brain', 'brain', 'eye blink', 'other', 'other', 'other', 'other', 'brain']
Overwriting existing file.
Writing ICA solution to /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20180209/eeg/sub-NZ000052_ses-20180209_task-restEyesClosed_proc-ica_ica.fif...
Writing '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20180209/eeg/sub-NZ000052_ses-20180209_task-restE

/tmp/ipykernel_406678/4062126790.py:30: RuntimeWarning: The provided Epochs instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  out = label_components(epochs, ica, method='iclabel')


Labels for sub-NZ000052, ses-20200728: ['brain', 'brain', 'brain', 'muscle artifact', 'brain', 'eye blink', 'brain', 'brain', 'brain', 'muscle artifact', 'brain', 'heart beat', 'heart beat', 'brain', 'heart beat', 'other', 'brain', 'other', 'muscle artifact', 'other', 'heart beat', 'channel noise', 'brain', 'eye blink', 'muscle artifact', 'brain', 'brain', 'muscle artifact', 'other', 'brain', 'other', 'brain', 'channel noise', 'channel noise', 'brain', 'other', 'brain', 'brain', 'channel noise', 'brain', 'other', 'channel noise', 'brain', 'other', 'brain', 'brain', 'muscle artifact', 'other', 'other', 'brain', 'other', 'brain', 'brain']
Overwriting existing file.
Writing ICA solution to /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20200728/eeg/sub-NZ000052_ses-20200728_task-restEyesClosed_proc-ica_ica.fif...
Writing '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-2020

...

Labels for sub-NZ084406, ses-20181126: ['channel noise', 'brain', 'other', 'brain', 'muscle artifact', 'muscle artifact', 'brain', 'brain', 'brain', 'brain', 'eye blink', 'brain', 'brain', 'channel noise', 'brain', 'brain', 'brain', 'brain', 'other', 'channel noise', 'other', 'other', 'brain', 'brain', 'other', 'brain', 'other', 'other', 'other', 'channel noise', 'channel noise', 'channel noise', 'channel noise', 'other', 'brain', 'channel noise', 'other', 'heart beat', 'brain', 'brain', 'brain', 'other', 'other', 'channel noise', 'other', 'other', 'brain', 'muscle artifact', 'channel noise', 'other', 'other', 'other', 'muscle artifact']
Overwriting existing file.
Writing ICA solution to /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ084406/ses-20181126/eeg/sub-NZ084406_ses-20181126_task-restEyesClosed_proc-ica_ica.fif...
Writing '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ084406/ses-201

/tmp/ipykernel_406678/4062126790.py:30: RuntimeWarning: The provided Epochs instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  out = label_components(epochs, ica, method='iclabel')


Labels for sub-NZ084406, ses-20200203: ['brain', 'heart beat', 'eye blink', 'heart beat', 'heart beat', 'muscle artifact', 'brain', 'brain', 'muscle artifact', 'brain', 'channel noise', 'muscle artifact', 'brain', 'brain', 'channel noise', 'brain', 'heart beat', 'muscle artifact', 'muscle artifact', 'eye blink', 'eye blink', 'muscle artifact', 'muscle artifact', 'channel noise', 'muscle artifact', 'muscle artifact', 'brain', 'muscle artifact', 'channel noise', 'muscle artifact', 'brain', 'other', 'channel noise', 'brain', 'brain', 'brain', 'muscle artifact', 'muscle artifact', 'other', 'channel noise', 'brain', 'other', 'channel noise', 'channel noise', 'brain', 'brain', 'other', 'other', 'brain', 'brain', 'brain', 'brain', 'channel noise', 'other', 'other']
Overwriting existing file.
Writing ICA solution to /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ084406/ses-20200203/eeg/sub-NZ084406_ses-20200203_task-restEyesClosed_proc-ica_ica.fi

/tmp/ipykernel_406678/4062126790.py:30: RuntimeWarning: The provided Epochs instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  out = label_components(epochs, ica, method='iclabel')


Labels for sub-NZ084501, ses-20171011: ['brain', 'brain', 'brain', 'brain', 'muscle artifact', 'eye blink', 'brain', 'channel noise', 'brain', 'brain', 'other', 'channel noise', 'other', 'other', 'channel noise', 'brain', 'other', 'brain', 'brain', 'muscle artifact', 'brain', 'brain', 'channel noise', 'channel noise', 'brain', 'brain', 'channel noise', 'channel noise', 'brain', 'other', 'other', 'brain', 'eye blink', 'channel noise', 'brain', 'brain', 'brain', 'brain', 'channel noise', 'channel noise', 'brain', 'brain', 'eye blink', 'other', 'brain', 'brain', 'other', 'brain', 'other']
Overwriting existing file.
Writing ICA solution to /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ084501/ses-20171011/eeg/sub-NZ084501_ses-20171011_task-restEyesClosed_proc-ica_ica.fif...
Writing '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ084501/ses-20171011/eeg/sub-NZ084501_ses-20171011_task-restEyesClos